In [1]:
# # --- Hardcode your file paths here -------------------------------------------
# PRED_PATH       = "/zpool/vladlab/active_drive/omaltz/scripts/geogaze/v3/resnet/test_masks/resnet50.a1_in1k_maskL_bc_bs_best/test_bc_bs_LR__model=resnet50.a1_in1k_maskL_bc_bs_best_mask.png"     # white object on black bg
# TARGET_PATH     = "/zpool/vladlab/active_drive/omaltz/scripts/geogaze/stimuli/out/test_stimuli/test_acc_masks/test_left_c_mask.png"         # black object on white bg
# DISTRACTOR_PATH = "/zpool/vladlab/active_drive/omaltz/scripts/geogaze/stimuli/out/test_stimuli/test_acc_masks/test_right_s_mask.png"        # black object on white bg

# THRESH_PRED = 128   # set None to auto
# THRESH_T    = 128
# THRESH_D    = 128

# import numpy as np
# from PIL import Image

# # ---- I/O + mask-safe resize ----
# def load_gray(path: str) -> np.ndarray:
#     return np.asarray(Image.open(path).convert("L"), dtype=np.float64)

# def nn_resize(arr: np.ndarray, new_shape) -> np.ndarray:
#     h, w = arr.shape; H, W = new_shape
#     if (H, W) == (h, w): 
#         return arr
#     r = (np.arange(H) * h) // H
#     c = (np.arange(W) * w) // W
#     return arr[np.ix_(r, c)]

# # ---- Binarize with polarity handling (True = object) ----
# def binarize(mask: np.ndarray, *, object_is_black: bool, thresh=None) -> np.ndarray:
#     m = mask
#     if thresh is None:
#         thresh = 0.5 * (float(m.min()) + float(m.max()))
#     return (m < thresh) if object_is_black else (m > thresh)

# # ---- Convert boolean mask to Nx2 coords [row, col] ----
# def coords_from_bool(mask_bool: np.ndarray) -> np.ndarray:
#     rr, cc = np.nonzero(mask_bool)
#     return np.stack([rr, cc], axis=1).astype(np.float32) if rr.size else np.empty((0,2), np.float32)

# # ---- ALL-PAIRS Euclidean distance sum  ----
# def distance_sum(A: np.ndarray, B: np.ndarray, max_bytes: int = 200_000_000) -> float:
#     """
#     Sum over all pairs (a in A, b in B) of Euclidean distance ||a-b||.
#     Returns 0.0 if either set is empty. Chunked to bound memory.
#     """
#     NA, NB = len(A), len(B)
#     if NA == 0 or NB == 0:
#         return 0.0

#     # ~200MB cap for a (k x NB) float32 chunk
#     k = max(1, int(max_bytes // (NB * 4)))

#     total = 0.0
#     B_sq = np.sum(B*B, axis=1, keepdims=True).T  # (1, NB)

#     for s in range(0, NA, k):
#         e = min(s + k, NA)
#         A_ch = A[s:e]                                # (k,2)
#         A_sq = np.sum(A_ch*A_ch, axis=1, keepdims=True)
#         d2 = A_sq + B_sq - 2.0*(A_ch @ B.T)
#         np.maximum(d2, 0.0, out=d2)
#         d = np.sqrt(d2, dtype=np.float32)            # Euclidean distances
#         total += float(d.sum())
#     return total

# # ---- Public API: distance-based all-pairs decision ----
# def allpairs_distance_choice(pred_img, target_img, distractor_img,
#                              pred_is_white_on_black=True, t_is_black_on_white=True, d_is_black_on_white=True,
#                              thresh_pred=None, thresh_t=None, thresh_d=None):
#     # 1) Binarize → True=object
#     P_bool = binarize(pred_img,   object_is_black=not pred_is_white_on_black, thresh=thresh_pred)
#     T_bool = binarize(target_img, object_is_black=t_is_black_on_white,         thresh=thresh_t)
#     D_bool = binarize(distractor_img, object_is_black=d_is_black_on_white,     thresh=thresh_d)

#     # 2) Coords
#     P = coords_from_bool(P_bool); T = coords_from_bool(T_bool); D = coords_from_bool(D_bool)

#     # 3) Edge case: no prediction pixels
#     if len(P) == 0:
#         return {
#             "distance_target": np.nan,
#             "distance_distractor": np.nan,
#             "closeness_target": np.nan,
#             "closeness_distractor": np.nan,
#             "accuracy": np.nan,
#             "counts": {"n_pred": 0, "n_target": len(T), "n_distractor": len(D)},
#             "choice": "NO_PREDICTION",
#         }

#     # 4) All-pairs Euclidean distance sums (smaller = closer)
#     D_t = distance_sum(T, P)
#     D_d = distance_sum(D, P)

#     denom = D_t + D_d
#     if denom > 0:
#         # closeness-to-target: 1 if much closer to target, 0 if much closer to distractor
#         closeness_target = D_d / denom
#     else:
#         closeness_target = 0.5  # ambiguous edge case

#     closeness_distractor = 1.0 - closeness_target

#     # Continuous "accuracy" = how target-like the prediction is (0–1)
#     accuracy = closeness_target

#     # Discrete label using distances (smaller distance = closer)
#     if D_t < D_d:
#         choice = "TARGET"
#     elif D_d < D_t:
#         choice = "DISTRACTOR"
#     else:
#         choice = "TIE"

#     return {
#         "distance_target": D_t,
#         "distance_distractor": D_d,
#         "closeness_target": closeness_target,           # 0–1, higher = closer to target than distractor
#         "closeness_distractor": closeness_distractor,
#         "accuracy": accuracy,                           # alias for closeness_target
#         "counts": {"n_pred": len(P), "n_target": len(T), "n_distractor": len(D)},
#         "choice": choice,
#     }

# # ---- Load, align, run ----
# pred = load_gray(PRED_PATH)
# targ = load_gray(TARGET_PATH)
# dist = load_gray(DISTRACTOR_PATH)

# pred = nn_resize(pred, targ.shape)
# dist = nn_resize(dist, targ.shape)

# res = allpairs_distance_choice(
#     pred_img=pred, target_img=targ, distractor_img=dist,
#     pred_is_white_on_black=True, t_is_black_on_white=True, d_is_black_on_white=True,
#     thresh_pred=THRESH_PRED, thresh_t=THRESH_T, thresh_d=THRESH_D,
# )

# print("Distance to TARGET:       ", res["distance_target"])
# print("Distance to DISTRACTOR:   ", res["distance_distractor"])
# print("Closeness to TARGET (0–1):", res["closeness_target"])
# print("Closeness to DIST (0–1):  ", res["closeness_distractor"])
# print("Continuous ACCURACY:      ", res["accuracy"])   # 0–1, higher = closer to TARGET
# print("CHOICE (discrete label):  ", res["choice"])


In [1]:
import os
import numpy as np
import pandas as pd
from PIL import Image

# ================== CONFIG (EDIT THESE) ==================
CSV_PATH = "/zpool/vladlab/active_drive/omaltz/scripts/geogaze/v3/geogaze_model_predictions.csv"

BASE_MASK_DIR   = "/zpool/vladlab/active_drive/omaltz/scripts/geogaze/stimuli/out/test_stimuli/test_acc_masks"
BASE_PRED_ROOT  = "/zpool/vladlab/active_drive/omaltz/scripts/geogaze/v3/resnet100/test_masks"

THRESH_PRED = 128   # set None to auto
THRESH_T    = 128
THRESH_D    = 128
# ========================================================

# ---- I/O + mask-safe resize ----
def load_gray(path: str) -> np.ndarray:
    return np.asarray(Image.open(path).convert("L"), dtype=np.float64)

def nn_resize(arr: np.ndarray, new_shape) -> np.ndarray:
    h, w = arr.shape
    H, W = new_shape
    if (H, W) == (h, w):
        return arr
    r = (np.arange(H) * h) // H
    c = (np.arange(W) * w) // W
    return arr[np.ix_(r, c)]

# ---- Binarize with polarity handling (True = object) ----
def binarize(mask: np.ndarray, *, object_is_black: bool, thresh=None) -> np.ndarray:
    m = mask
    if thresh is None:
        thresh = 0.5 * (float(m.min()) + float(m.max()))
    return (m < thresh) if object_is_black else (m > thresh)

# ---- Convert boolean mask to Nx2 coords [row, col] ----
def coords_from_bool(mask_bool: np.ndarray) -> np.ndarray:
    rr, cc = np.nonzero(mask_bool)
    return np.stack([rr, cc], axis=1).astype(np.float32) if rr.size else np.empty((0, 2), np.float32)

# ---- ALL-PAIRS Euclidean distance sum (chunked) ----
def distance_sum(A: np.ndarray, B: np.ndarray, max_bytes: int = 200_000_000) -> float:
    """
    Sum over all pairs (a in A, b in B) of Euclidean distance ||a-b||.
    Returns 0.0 if either set is empty. Chunked to bound memory.
    """
    NA, NB = len(A), len(B)
    if NA == 0 or NB == 0:
        return 0.0

    # ~200MB cap for a (k x NB) float32 chunk
    k = max(1, int(max_bytes // (NB * 4)))

    total = 0.0
    B_sq = np.sum(B * B, axis=1, keepdims=True).T  # (1, NB)

    for s in range(0, NA, k):
        e = min(s + k, NA)
        A_ch = A[s:e]                                # (k,2)
        A_sq = np.sum(A_ch * A_ch, axis=1, keepdims=True)
        d2 = A_sq + B_sq - 2.0 * (A_ch @ B.T)
        np.maximum(d2, 0.0, out=d2)
        d = np.sqrt(d2, dtype=np.float32)            # Euclidean distances
        total += float(d.sum())
    return total

# ---- Public API: distance-based all-pairs continuous score ----
def allpairs_distance_score(pred_img, target_img, distractor_img,
                            pred_is_white_on_black=True, t_is_black_on_white=True, d_is_black_on_white=True,
                            thresh_pred=None, thresh_t=None, thresh_d=None):
    # 1) Binarize → True=object
    P_bool = binarize(pred_img,   object_is_black=not pred_is_white_on_black, thresh=thresh_pred)
    T_bool = binarize(target_img, object_is_black=t_is_black_on_white,        thresh=thresh_t)
    D_bool = binarize(distractor_img, object_is_black=d_is_black_on_white,    thresh=thresh_d)

    # 2) Coords
    P = coords_from_bool(P_bool)
    T = coords_from_bool(T_bool)
    D = coords_from_bool(D_bool)

    # 3) Edge case: no prediction pixels → treat as neutral (0.5)
    if len(P) == 0:
        return {
            "distance_target": np.nan,
            "distance_distractor": np.nan,
            "accuracy": 0.5,  # neutral between target (1) and distractor (0)
            "counts": {"n_pred": 0, "n_target": len(T), "n_distractor": len(D)},
            "choice": "NO_PREDICTION",
        }

    # 4) All-pairs Euclidean distance sums (smaller = closer)
    D_t = distance_sum(T, P)
    D_d = distance_sum(D, P)

    denom = D_t + D_d
    if denom > 0:
        # Our definition: closeness-to-target (0–1, higher = closer to target)
        # accuracy = D_d / (D_t + D_d)
        accuracy = D_d / denom
    else:
        # Degenerate case → also treat as neutral
        accuracy = 0.5

    # Discrete choice if you still want it
    if D_t < D_d:
        choice = "TARGET"
    elif D_d < D_t:
        choice = "DISTRACTOR"
    else:
        choice = "TIE"

    return {
        "distance_target": D_t,
        "distance_distractor": D_d,
        "accuracy": accuracy,  # 0–1, higher = more target-like
        "counts": {"n_pred": len(P), "n_target": len(T), "n_distractor": len(D)},
        "choice": choice,
    }

# ---- Helper to build full paths for a row ----
def build_paths_for_row(row):
    target_path     = os.path.join(BASE_MASK_DIR, row["target"])
    distractor_path = os.path.join(BASE_MASK_DIR, row["distractor"])
    model_folder    = f"resnet50.a1_in1k_{row['model']}_best"
    pred_path       = os.path.join(BASE_PRED_ROOT, model_folder, row["prediction"])
    return pred_path, target_path, distractor_path

# ================== RUN OVER CSV ==================
df = pd.read_csv(CSV_PATH)

for idx, row in df.iterrows():
    try:
        pred_path, targ_path, dist_path = build_paths_for_row(row)

        # Load
        pred = load_gray(pred_path)
        targ = load_gray(targ_path)
        dist = load_gray(dist_path)

        # Align sizes (match target shape)
        pred = nn_resize(pred, targ.shape)
        dist = nn_resize(dist, targ.shape)

        # Run metric
        res = allpairs_distance_score(
            pred_img=pred,
            target_img=targ,
            distractor_img=dist,
            pred_is_white_on_black=True,
            t_is_black_on_white=True,
            d_is_black_on_white=True,
            thresh_pred=THRESH_PRED,
            thresh_t=THRESH_T,
            thresh_d=THRESH_D,
        )

        acc = res["accuracy"]

    except Exception as e:
        print(f"Row {idx}: ERROR ({e})")
        acc = np.nan  # still NaN for true errors (missing file, etc.)

    # write continuous accuracy into a new column
    df.at[idx, "acc_continuous"] = acc

# Save out new CSV with continuous accuracy filled
out_path = os.path.splitext(CSV_PATH)[0] + "_with_acc_11_20_25_resnet100.csv"
df.to_csv(out_path, index=False)
print(f"Done. Saved updated CSV to:\n{out_path}")


Done. Saved updated CSV to:
/zpool/vladlab/active_drive/omaltz/scripts/geogaze/v3/geogaze_model_predictions_with_acc_11_20_25_resnet100.csv


In [3]:
### DICE COEFFICENT ACCURASY

import os
import numpy as np
import pandas as pd
from PIL import Image

# ================== CONFIG (EDIT THESE) ==================
CSV_PATH = "/zpool/vladlab/active_drive/omaltz/scripts/geogaze/v3/geogaze_model_predictions.csv"

BASE_MASK_DIR   = "/zpool/vladlab/active_drive/omaltz/scripts/geogaze/stimuli/out/test_stimuli/test_acc_masks"
BASE_PRED_ROOT  = "/zpool/vladlab/active_drive/omaltz/scripts/geogaze/v3/resnet100/test_masks"

THRESH_PRED = 128   # set None to auto
THRESH_T    = 128
THRESH_D    = 128
# ========================================================

# ---- I/O + mask-safe resize ----
def load_gray(path: str) -> np.ndarray:
    """Load an image as grayscale float64 numpy array."""
    return np.asarray(Image.open(path).convert("L"), dtype=np.float64)

def nn_resize(arr: np.ndarray, new_shape) -> np.ndarray:
    """Nearest-neighbor resize for 2D arrays (H,W)."""
    h, w = arr.shape
    H, W = new_shape
    if (H, W) == (h, w):
        return arr
    r = (np.arange(H) * h) // H
    c = (np.arange(W) * w) // W
    return arr[np.ix_(r, c)]

# ---- Binarize with polarity handling (True = object) ----
def binarize(mask: np.ndarray, *, object_is_black: bool, thresh=None) -> np.ndarray:
    """
    Convert grayscale mask into boolean mask where True = object pixel.

    object_is_black:
        True  -> object dark on light background -> use m < thresh
        False -> object light on dark background -> use m > thresh
    """
    m = mask
    if thresh is None:
        thresh = 0.5 * (float(m.min()) + float(m.max()))
    return (m < thresh) if object_is_black else (m > thresh)

# ---- Dice coefficient for boolean masks ----
def dice_from_bool(mask1: np.ndarray, mask2: np.ndarray) -> float:
    """
    Dice coefficient between two boolean masks (True = object pixel).

    Returns 0.0 if both masks are empty.
    """
    m1 = mask1.astype(bool)
    m2 = mask2.astype(bool)

    inter = np.logical_and(m1, m2).sum()
    size1 = m1.sum()
    size2 = m2.sum()

    denom = size1 + size2
    if denom == 0:
        # both empty -> define Dice = 0
        return 0.0

    return 2.0 * inter / denom

# ---- Public API: Dice-based continuous score ----
def allpairs_dice_score(pred_img, target_img, distractor_img,
                        pred_is_white_on_black=True,
                        t_is_black_on_white=True,
                        d_is_black_on_white=True,
                        thresh_pred=None, thresh_t=None, thresh_d=None):
    """
    Compute Dice(pred, target) and Dice(pred, distractor), then form a
    continuous accuracy score in [0,1]:

        acc = Dice(pred, target) / (Dice(pred, target) + Dice(pred, distractor))

    Higher acc = more target-like.
    """

    # 1) Binarize -> True = object pixel
    P_bool = binarize(pred_img,   object_is_black=not pred_is_white_on_black, thresh=thresh_pred)
    T_bool = binarize(target_img, object_is_black=t_is_black_on_white,        thresh=thresh_t)
    D_bool = binarize(distractor_img, object_is_black=d_is_black_on_white,    thresh=thresh_d)

    # 2) Edge case: no prediction pixels -> neutral
    n_pred = P_bool.sum()
    n_targ = T_bool.sum()
    n_dist = D_bool.sum()

    if n_pred == 0:
        return {
            "dice_target": np.nan,
            "dice_distractor": np.nan,
            "accuracy": 0.5,  # neutral between target (1) and distractor (0)
            "counts": {"n_pred": int(n_pred), "n_target": int(n_targ), "n_distractor": int(n_dist)},
            "choice": "NO_PREDICTION",
        }

    # 3) Dice overlaps
    dice_t = dice_from_bool(P_bool, T_bool)
    dice_d = dice_from_bool(P_bool, D_bool)

    # 4) Continuous accuracy via ratio of Dice scores
    denom = dice_t + dice_d
    if denom > 0:
        accuracy = dice_t / denom  # higher = prediction overlaps target more than distractor
    else:
        # no overlap with either -> treat as neutral
        accuracy = 0.5

    # 5) Discrete choice (if you still want it)
    if dice_t > dice_d:
        choice = "TARGET"
    elif dice_d > dice_t:
        choice = "DISTRACTOR"
    else:
        choice = "TIE"

    return {
        "dice_target": dice_t,
        "dice_distractor": dice_d,
        "accuracy": accuracy,
        "counts": {"n_pred": int(n_pred), "n_target": int(n_targ), "n_distractor": int(n_dist)},
        "choice": choice,
    }

# ---- Helper to build full paths for a row ----
def build_paths_for_row(row):
    target_path     = os.path.join(BASE_MASK_DIR, row["target"])
    distractor_path = os.path.join(BASE_MASK_DIR, row["distractor"])
    model_folder    = f"resnet50.a1_in1k_{row['model']}_best"
    pred_path       = os.path.join(BASE_PRED_ROOT, model_folder, row["prediction"])
    return pred_path, target_path, distractor_path

# ================== RUN OVER CSV ==================
df = pd.read_csv(CSV_PATH)

for idx, row in df.iterrows():
    try:
        pred_path, targ_path, dist_path = build_paths_for_row(row)

        # Load
        pred = load_gray(pred_path)
        targ = load_gray(targ_path)
        dist = load_gray(dist_path)

        # Align sizes (match target shape)
        pred = nn_resize(pred, targ.shape)
        dist = nn_resize(dist, targ.shape)

        # Run Dice-based metric
        res = allpairs_dice_score(
            pred_img=pred,
            target_img=targ,
            distractor_img=dist,
            pred_is_white_on_black=True,
            t_is_black_on_white=True,
            d_is_black_on_white=True,
            thresh_pred=THRESH_PRED,
            thresh_t=THRESH_T,
            thresh_d=THRESH_D,
        )

        acc = res["accuracy"]

    except Exception as e:
        print(f"Row {idx}: ERROR ({e})")
        acc = np.nan  # NaN for true errors (missing file, etc.)

    # write continuous accuracy into a new column
    df.at[idx, "acc_continuous"] = acc

# Save out new CSV with continuous accuracy filled
out_path = os.path.splitext(CSV_PATH)[0] + "_resnet100_dice_12_3_25.csv"
df.to_csv(out_path, index=False)
print(f"Done. Saved updated CSV to:\n{out_path}")


Done. Saved updated CSV to:
/zpool/vladlab/active_drive/omaltz/scripts/geogaze/v3/geogaze_model_predictions_resnet100_dice_12_3_25.csv
